# Data Cleaning

**Mục đích:** Thực hiện cleaning theo checklist từ EDA, chuẩn bị dữ liệu cho bước Data Preparing & Modeling.

**Quy trình:** Requirement → Context Analysis → Implementation → Review → Evaluation

---

## 0. Thiết lập & Cấu hình

Định nghĩa paths, constants, và helper functions dùng chung cho toàn bộ notebook.

In [1]:
import pandas as pd
import numpy as np
import json
import re
import os
from pathlib import Path
from scipy import stats

# ============================================
# CONFIGURATION
# ============================================
BASE_DIR = Path("../Dataset")
OUTPUT_DIR = Path("../Dataset/cleaned")
OUTPUT_DIR.mkdir(exist_ok=True)

# CBC artifact placeholder (NHANES)
CBC_ARTIFACT = "5.397605346934028e-79"

# Physiological plausibility bounds (adult reference)
PLAUSIBILITY = {
    "Hemoglobin": (7.0, 20.0),
    "RBC": (2.0, 7.0),
    "WBC": (2000, 50000),
    "AST": (0, 500),
    "ALT": (0, 500),
    "Cholestrol": (50, 400),
    "Spirometry": (0.5, 8.0),
    "Creatinine": (0.3, 15.0),
    "Glucose": (30, 500),
    "Lipase": (0, 500),
    "Troponin": (0, 10),
}

# ============================================
# LOGGING HELPER
# ============================================
class CleaningLogger:
    def __init__(self, dataset_name):
        self.dataset_name = dataset_name
        self.log = []

    def info(self, msg):
        self.log.append(f"[INFO] {msg}")
        print(f"[INFO] {msg}")

    def warn(self, msg):
        self.log.append(f"[WARN] {msg}")
        print(f"[WARN] {msg}")

    def error(self, msg):
        self.log.append(f"[ERROR] {msg}")
        print(f"[ERROR] {msg}")

    def action(self, msg):
        self.log.append(f"[ACTION] {msg}")
        print(f"[ACTION] {msg}")

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"# Cleaning Log - {self.dataset_name}\n\n")
            for entry in self.log:
                f.write(entry + "\n")

print("Configuration loaded.")

Configuration loaded.


---

## 1. CBC (Complete Blood Count)

### Đề xuất cleaning từ EDA:
- Artifact handling: thay thế toàn bộ `5.397605346934028e-79` bằng `NaN`
- Numeric coercion: ép kiểu numeric với `errors='coerce'`
- Outlier: flag bằng biến nhị phân thay vì xóa
- Missing imputation: để dành cho bước preparing (MICE/KNN)

In [2]:
# ============================================
# 1. CBC CLEANING
# ============================================
logger = CleaningLogger("CBC")

df_cbc = pd.read_csv(BASE_DIR / "CBC.csv")
logger.info(f"Loaded CBC: {df_cbc.shape[0]} rows x {df_cbc.shape[1]} columns")

# 1. Artifact sanitization
artifact_mask = df_cbc.apply(
    lambda row: row.astype(str).str.contains(CBC_ARTIFACT, na=False).any(),
    axis=1
)
artifact_rows = artifact_mask.sum()
logger.warn(f"Artifact rows detected: {artifact_rows} ({artifact_rows/len(df_cbc)*100:.1f}%)")

# Replace artifact with NaN (in-place tracking)
artifact_cells_before = 0
for col in df_cbc.columns:
    cnt = df_cbc[col].astype(str).str.contains(CBC_ARTIFACT, na=False).sum()
    if cnt > 0:
        artifact_cells_before += cnt
        df_cbc[col] = df_cbc[col].replace(CBC_ARTIFACT, np.nan)

logger.action(f"Replaced {artifact_cells_before} artifact cells with NaN")

# 2. Numeric coercion
numeric_cols = [c for c in df_cbc.columns if c != "SEQN"]
coerce_count = 0
for col in numeric_cols:
    original_dtype = df_cbc[col].dtype
    df_cbc[col] = pd.to_numeric(df_cbc[col], errors="coerce")
    # Count newly coerced NaNs (simple heuristic: compare non-null counts)

logger.action(f"Coerced {len(numeric_cols)} numeric columns to float64")

# 3. Outlier flagging (IQR-based, per column)
OUTLIER_PREFIX = "outlier_"
for col in numeric_cols:
    clean = df_cbc[col].dropna()
    if len(clean) < 10:
        continue
    q1, q3 = clean.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        continue
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = (df_cbc[col] < lower) | (df_cbc[col] > upper)
    df_cbc[f"{OUTLIER_PREFIX}{col}"] = outlier_mask.astype(int)

logger.action(f"Flagged outliers in {len(numeric_cols)} columns with prefix '{OUTLIER_PREFIX}'")

# 4. Missing summary
missing_summary = df_cbc.isna().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(df_cbc) * 100).round(2)

logger.info("Missing value summary (top 10):")
for col in missing_summary.head(10).index:
    logger.info(f"  {col}: {missing_summary[col]} ({missing_pct[col]:.1f}%)")

# 5. Validation checks
assert df_cbc["SEQN"].is_unique, "SEQN is not unique"
logger.info("Validation passed: SEQN is unique")

# 6. Save cleaned data
output_path = OUTPUT_DIR / "CBC_cleaned.csv"
df_cbc.to_csv(output_path, index=False)
logger.action(f"Saved cleaned data to {output_path}")
logger.info(f"Final shape: {df_cbc.shape}")

# Summary for review
print("\n" + "="*60)
print("CBC CLEANING SUMMARY")
print("="*60)
print(f"Original rows: {df_cbc.shape[0]}")
print(f"Artifact cells replaced: {artifact_cells_before}")
print(f"Outlier flags added: {sum(1 for c in df_cbc.columns if c.startswith(OUTLIER_PREFIX))}")
print(f"Output: {output_path}")
print("="*60)

[INFO] Loaded CBC: 9165 rows x 21 columns
[WARN] Artifact rows detected: 3961 (43.2%)
[ACTION] Replaced 4284 artifact cells with NaN
[ACTION] Coerced 20 numeric columns to float64
[ACTION] Flagged outliers in 20 columns with prefix 'outlier_'
[INFO] Missing value summary (top 10):
[INFO]   LBXLYPCT: 1049 (11.4%)
[INFO]   LBDLYMNO: 1049 (11.4%)
[INFO]   LBXMOPCT: 1049 (11.4%)
[INFO]   LBXNEPCT: 1049 (11.4%)
[INFO]   LBXEOPCT: 1049 (11.4%)
[INFO]   LBDMONO: 1049 (11.4%)
[INFO]   LBXBAPCT: 1049 (11.4%)
[INFO]   LBDEONO: 1049 (11.4%)
[INFO]   LBDNENO: 1049 (11.4%)
[INFO]   LBDBANO: 1049 (11.4%)
[INFO] Validation passed: SEQN is unique
[ACTION] Saved cleaned data to ..\Dataset\cleaned\CBC_cleaned.csv
[INFO] Final shape: (9165, 41)

CBC CLEANING SUMMARY
Original rows: 9165
Artifact cells replaced: 4284
Outlier flags added: 20
Output: ..\Dataset\cleaned\CBC_cleaned.csv


---

## 2. Diseases_and_Symptoms

### Đề xuất cleaning từ EDA:
- Loại bỏ quasi-constant features (variance threshold < 0.01)
- Validate binary values (chỉ cho phép 0/1)
- Flag class imbalance (không resample ở cleaning step)
- Lưu danh sách features giữ lại để modeling

In [3]:
# ============================================
# 2. DISEASES_AND_SYMPTOMS CLEANING
# ============================================
logger = CleaningLogger("Diseases_and_Symptoms")

df_dis = pd.read_csv(BASE_DIR / "Diseases_and_Symptoms.csv")
logger.info(f"Loaded Diseases_and_Symptoms: {df_dis.shape[0]} rows x {df_dis.shape[1]} columns")

target_col = "diseases"
feature_cols = [c for c in df_dis.columns if c != target_col]

# 1. Validate binary values
invalid_counts = {}
for col in feature_cols:
    unique_vals = df_dis[col].dropna().unique()
    invalid = [v for v in unique_vals if v not in [0, 1, 0.0, 1.0]]
    if invalid:
        invalid_counts[col] = invalid

if invalid_counts:
    logger.warn(f"Found non-binary values in {len(invalid_counts)} columns")
    for col, vals in list(invalid_counts.items())[:5]:
        logger.warn(f"  {col}: {vals}")
    # Coerce to numeric, invalid -> NaN
    for col in feature_cols:
        df_dis[col] = pd.to_numeric(df_dis[col], errors="coerce")
    logger.action("Coerced non-binary values to NaN")
else:
    logger.info("All feature columns are binary (0/1).")

# 2. Remove quasi-constant features (variance < 0.01)
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
X = df_dis[feature_cols].fillna(0)  # fillna for selector
selector.fit(X)
keep_mask = selector.get_support()
keep_cols = [col for col, keep in zip(feature_cols, keep_mask) if keep]
drop_cols = [col for col, keep in zip(feature_cols, keep_mask) if not keep]

logger.warn(f"Quasi-constant features to drop: {len(drop_cols)}")
if drop_cols:
    for col in drop_cols[:10]:
        logger.warn(f"  {col}")
    if len(drop_cols) > 10:
        logger.warn(f"  ... and {len(drop_cols)-10} more")

df_dis_clean = df_dis[[target_col] + keep_cols].copy()
logger.action(f"Kept {len(keep_cols)} features, dropped {len(drop_cols)} quasi-constant features")

# 3. Class imbalance report (flag only)
class_counts = df_dis_clean[target_col].value_counts()
imbalance_ratio = class_counts.iloc[0] / class_counts.iloc[-1]
logger.warn(f"Class imbalance ratio (max/min): {imbalance_ratio:.1f}x")
logger.info("Top 5 classes:")
for cls, cnt in class_counts.head(5).items():
    logger.info(f"  {cls}: {cnt} ({cnt/len(df_dis_clean)*100:.1f}%)")

# 4. Validation checks
assert df_dis_clean[target_col].notna().all(), "Target has missing values"
logger.info("Validation passed: Target column has no missing values")

# 5. Save
output_path = OUTPUT_DIR / "Diseases_and_Symptoms_cleaned.csv"
df_dis_clean.to_csv(output_path, index=False)
logger.action(f"Saved cleaned data to {output_path}")

# Save feature list for preparing step
feature_list_path = OUTPUT_DIR / "Diseases_and_Symptoms_features.json"
with open(feature_list_path, "w") as f:
    json.dump({
        "target": target_col,
        "features": keep_cols,
        "n_classes": len(class_counts),
        "imbalance_ratio": round(float(imbalance_ratio), 2)
    }, f, indent=2)
logger.action(f"Saved feature metadata to {feature_list_path}")

print("\n" + "="*60)
print("DISEASES_AND_SYMPTOMS CLEANING SUMMARY")
print("="*60)
print(f"Original features: {len(feature_cols)}")
print(f"Kept features: {len(keep_cols)}")
print(f"Dropped quasi-constant: {len(drop_cols)}")
print(f"Classes: {len(class_counts)}")
print(f"Imbalance ratio: {imbalance_ratio:.1f}x")
print(f"Output: {output_path}")
print("="*60)

[INFO] Loaded Diseases_and_Symptoms: 246945 rows x 378 columns
[INFO] All feature columns are binary (0/1).
[WARN] Quasi-constant features to drop: 231
[WARN]   breathing fast
[WARN]   throat swelling
[WARN]   lump in throat
[WARN]   throat feels tight
[WARN]   groin mass
[WARN]   emotional symptoms
[WARN]   elbow weakness
[WARN]   back weakness
[WARN]   pus in sputum
[WARN]   symptoms of the scrotum and testes
[WARN]   ... and 221 more
[ACTION] Kept 146 features, dropped 231 quasi-constant features
[WARN] Class imbalance ratio (max/min): 1219.0x
[INFO] Top 5 classes:
[INFO]   cystitis: 1219 (0.5%)
[INFO]   vulvodynia: 1218 (0.5%)
[INFO]   nose disorder: 1218 (0.5%)
[INFO]   complex regional pain syndrome: 1217 (0.5%)
[INFO]   spondylosis: 1216 (0.5%)
[INFO] Validation passed: Target column has no missing values
[ACTION] Saved cleaned data to ..\Dataset\cleaned\Diseases_and_Symptoms_cleaned.csv
[ACTION] Saved feature metadata to ..\Dataset\cleaned\Diseases_and_Symptoms_features.json

D

---

## 3. Laboratory Data

### Đề xuất cleaning từ EDA:
- Fix trailing space: `Disease ` → `Disease`
- Numeric coercion + validate physiological ranges
- Flag outliers thay vì xóa
- One-Hot encoding cho `Gender`
- Missing imputation để dành cho preparing step

In [4]:
# ============================================
# 3. LABORATORY DATA CLEANING
# ============================================
logger = CleaningLogger("Laboratory_Data")

df_lab = pd.read_csv(BASE_DIR / "laboratory_data.csv")
logger.info(f"Loaded Laboratory Data: {df_lab.shape[0]} rows x {df_lab.shape[1]} columns")

# 1. Fix trailing space in target column
if "Disease " in df_lab.columns:
    df_lab = df_lab.rename(columns={"Disease ": "Disease"})
    logger.action("Renamed 'Disease ' -> 'Disease' (removed trailing space)")

target_col = "Disease"
feature_cols = [c for c in df_lab.columns if c != target_col]

# 2. Separate types
numeric_cols = df_lab[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_lab[feature_cols].select_dtypes(include=["object"]).columns.tolist()

logger.info(f"Numeric features: {numeric_cols}")
logger.info(f"Categorical features: {cat_cols}")

# 3. Numeric coercion
for col in numeric_cols:
    df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

logger.action(f"Coerced {len(numeric_cols)} numeric columns")

# 4. Physiological plausibility flagging
OUTLIER_PREFIX = "plausibility_outlier_"
for col, (low, high) in PLAUSIBILITY.items():
    if col not in df_lab.columns:
        continue
    mask = (df_lab[col] < low) | (df_lab[col] > high)
    df_lab[f"{OUTLIER_PREFIX}{col}"] = mask.astype(int)

logger.action(f"Flagged physiological outliers for {len(PLAUSIBILITY)} features")

# 5. IQR outlier flagging (per disease group)
IQR_PREFIX = "iqr_outlier_"
key_features = ["Hemoglobin", "WBC", "Glucose", "Troponin", "Creatinine"]
for feat in key_features:
    if feat not in df_lab.columns:
        continue
    outlier_flags = pd.Series(0, index=df_lab.index)
    for disease in df_lab[target_col].dropna().unique():
        sub = df_lab[df_lab[target_col] == disease][feat].dropna()
        if len(sub) < 10:
            continue
        q1, q3 = sub.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        idx = df_lab[(df_lab[target_col] == disease)].index
        outlier_flags.loc[idx] = ((df_lab.loc[idx, feat] < lower) | (df_lab.loc[idx, feat] > upper)).astype(int)
    df_lab[f"{IQR_PREFIX}{feat}"] = outlier_flags

logger.action(f"Flagged IQR outliers for key features: {key_features}")

# 6. Categorical encoding: Gender -> One-Hot
if "Gender" in df_lab.columns:
    gender_dummies = pd.get_dummies(df_lab["Gender"], prefix="Gender", dummy_na=False)
    df_lab = pd.concat([df_lab.drop(columns=["Gender"]), gender_dummies], axis=1)
    logger.action("One-Hot encoded 'Gender' column")

# 7. Missing summary
missing_summary = df_lab.isna().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(df_lab) * 100).round(2)

logger.info("Missing value summary:")
for col in missing_summary[missing_summary > 0].index:
    logger.info(f"  {col}: {missing_summary[col]} ({missing_pct[col]:.1f}%)")

# 8. Validation checks
assert df_lab[target_col].notna().all(), "Target has missing values"
logger.info("Validation passed: Target column has no missing values")

# 9. Save
output_path = OUTPUT_DIR / "Laboratory_Data_cleaned.csv"
df_lab.to_csv(output_path, index=False)
logger.action(f"Saved cleaned data to {output_path}")

print("\n" + "="*60)
print("LABORATORY DATA CLEANING SUMMARY")
print("="*60)
print(f"Original rows: {df_lab.shape[0]}")
print(f"Columns after cleaning: {df_lab.shape[1]}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features (before encoding): {len(cat_cols)}")
print(f"Output: {output_path}")
print("="*60)

[INFO] Loaded Laboratory Data: 12009 rows x 14 columns
[ACTION] Renamed 'Disease ' -> 'Disease' (removed trailing space)
[INFO] Numeric features: ['Age', 'Hemoglobin', 'RBC', 'WBC', 'AST (aspartate aminotransferase)', 'ALT (alanine aminotransferase)', 'Cholestrol', 'Spirometry', 'Creatinine', 'Glucose', 'Lipase', 'Troponin']
[INFO] Categorical features: ['Gender']
[ACTION] Coerced 12 numeric columns
[ACTION] Flagged physiological outliers for 11 features
[ACTION] Flagged IQR outliers for key features: ['Hemoglobin', 'WBC', 'Glucose', 'Troponin', 'Creatinine']
[ACTION] One-Hot encoded 'Gender' column
[INFO] Missing value summary:
[INFO] Validation passed: Target column has no missing values
[ACTION] Saved cleaned data to ..\Dataset\cleaned\Laboratory_Data_cleaned.csv

C:\Users\hotro\AppData\Local\Temp\ipykernel_16520\3756068656.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_lab[feature_cols].select_dtypes(include=["object"]).columns.tolist()




LABORATORY DATA CLEANING SUMMARY
Original rows: 12009
Columns after cleaning: 29
Numeric features: 12
Categorical features (before encoding): 1
Output: ..\Dataset\cleaned\Laboratory_Data_cleaned.csv


---

## 4. Personalized Medication Dataset

### Đề xuất cleaning từ EDA:
- Kiểm tra BMI mismatch với Height/Weight
- Tách cột hỗn hợp: `Dosage` → `Dosage_Value` + `Dosage_Unit`, `Duration` → `Duration_Days` + `Duration_Unit`
- Tách multi-value columns thành binary indicators
- Xử lý semantics của "None" (giữ như category)
- Drop leakage columns: `Treatment_Effectiveness`, `Adverse_Reactions`

In [5]:
# ============================================
# 4. PERSONALIZED MEDICATION CLEANING
# ============================================
logger = CleaningLogger("Personalized_Medication")

df_med = pd.read_csv(BASE_DIR / "personalized_medication_dataset.csv")
target_col = "Recommended_Medication"
df_med[target_col] = df_med[target_col].fillna("None")
logger.info(f"Loaded Personalized Medication: {df_med.shape[0]} rows x {df_med.shape[1]} columns")


# 1. BMI consistency check
if all(c in df_med.columns for c in ["Height_cm", "Weight_kg", "BMI"]):
    df_med["BMI_calculated"] = df_med["Weight_kg"] / (df_med["Height_cm"] / 100) ** 2
    df_med["BMI_diff"] = (df_med["BMI"] - df_med["BMI_calculated"]).abs()
    mismatch = df_med[df_med["BMI_diff"] > 1.0]
    logger.warn(f"BMI mismatch > 1.0: {len(mismatch)} rows ({len(mismatch)/len(df_med)*100:.1f}%)")
    if len(mismatch) > 0:
        logger.info("Sample mismatches:")
        for _, row in mismatch.head(3).iterrows():
            logger.info(f"  {row['Patient_ID']}: BMI={row['BMI']}, calc={row['BMI_calculated']:.1f}, diff={row['BMI_diff']:.1f}")
    # Use calculated BMI to reduce rounding error
    df_med["BMI"] = df_med["BMI_calculated"].round(1)
    df_med = df_med.drop(columns=["BMI_calculated", "BMI_diff"])
    logger.action("Replaced BMI with calculated value (rounded 1 decimal)")

# 2. Split Dosage -> Dosage_Value + Dosage_Unit
if "Dosage" in df_med.columns:
    dosage_extracted = df_med["Dosage"].astype(str).str.extract(r"(?P<value>\d+\.?\d*)\s*(?P<unit>\w*)")
    df_med["Dosage_Value"] = pd.to_numeric(dosage_extracted["value"], errors="coerce")
    df_med["Dosage_Unit"] = dosage_extracted["unit"].replace("", "None")
    df_med = df_med.drop(columns=["Dosage"])
    logger.action("Split 'Dosage' into 'Dosage_Value' and 'Dosage_Unit'")

# 3. Split Duration -> Duration_Days + Duration_Unit
if "Duration" in df_med.columns:
    duration_extracted = df_med["Duration"].astype(str).str.extract(r"(?P<value>\d+\.?\d*)\s*(?P<unit>\w*)")
    df_med["Duration_Days"] = pd.to_numeric(duration_extracted["value"], errors="coerce")
    df_med["Duration_Unit"] = duration_extracted["unit"].replace("", "None")
    df_med = df_med.drop(columns=["Duration"])
    logger.action("Split 'Duration' into 'Duration_Days' and 'Duration_Unit'")

# 4. Split multi-value string columns into binary indicators
MULTI_VALUE_COLS = ["Symptoms", "Drug_Allergies", "Chronic_Conditions", "Genetic_Disorders"]
for col in MULTI_VALUE_COLS:
    if col not in df_med.columns:
        continue
    df_med[col] = df_med[col].fillna("None")
    # Split by comma, explode, get dummies, groupby max
    s = df_med[col].astype(str).str.split(",")
    exploded = s.explode()
    exploded = exploded.str.strip()
    dummies = pd.get_dummies(exploded, prefix=col)
    dummies = dummies.groupby(level=0).max()
    # Align index
    dummies = dummies.reindex(df_med.index, fill_value=0)
    df_med = pd.concat([df_med.drop(columns=[col]), dummies], axis=1)
    logger.action(f"Split '{col}' into {dummies.shape[1]} binary indicator columns")

# 5. Handle 'None' semantics: keep as category, don't impute
NONE_COLS = ["Drug_Allergies", "Genetic_Disorders", "Dosage_Unit", "Duration_Unit", "Adverse_Reactions"]
for col in NONE_COLS:
    if col in df_med.columns:
        df_med[col] = df_med[col].fillna("None").astype(str)
        # Ensure it's string type for consistent encoding later
        df_med[col] = df_med[col].astype(str)

logger.action("Preserved 'None' as explicit category in relevant columns")

# 6. Drop leakage columns (if model only predicts medication)
LEAKAGE_COLS = ["Treatment_Effectiveness", "Adverse_Reactions"]
existing_leakage = [c for c in LEAKAGE_COLS if c in df_med.columns]
if existing_leakage:
    df_med = df_med.drop(columns=existing_leakage)
    logger.action(f"Dropped leakage columns: {existing_leakage}")

# 7. Numeric coercion for remaining numeric columns
remaining_numeric = df_med.select_dtypes(include=[np.number]).columns.tolist()
for col in remaining_numeric:
    df_med[col] = pd.to_numeric(df_med[col], errors="coerce")

logger.action(f"Coerced {len(remaining_numeric)} numeric columns")

# 8. Validation checks
assert df_med[target_col].notna().all(), "Target has missing values"
logger.info("Validation passed: Target column has no missing values")

if "Patient_ID" in df_med.columns:
    assert df_med["Patient_ID"].is_unique, "Patient_ID is not unique"
    logger.info("Validation passed: Patient_ID is unique")

# 9. Save
output_path = OUTPUT_DIR / "Personalized_Medication_cleaned.csv"
df_med.to_csv(output_path, index=False)
logger.action(f"Saved cleaned data to {output_path}")

print("\n" + "="*60)
print("PERSONALIZED MEDICATION CLEANING SUMMARY")
print("="*60)
print(f"Original rows: {df_med.shape[0]}")
print(f"Columns after cleaning: {df_med.shape[1]}")
print(f"Target: {target_col}")
print(f"Output: {output_path}")
print("="*60)

[INFO] Loaded Personalized Medication: 1000 rows x 17 columns
[WARN] BMI mismatch > 1.0: 916 rows (91.6%)
[INFO] Sample mismatches:
[INFO]   P0001: BMI=21.1, calc=23.0, diff=1.9
[INFO]   P0002: BMI=30.2, calc=23.7, diff=6.5
[INFO]   P0003: BMI=27.0, calc=30.8, diff=3.8
[ACTION] Replaced BMI with calculated value (rounded 1 decimal)
[ACTION] Split 'Dosage' into 'Dosage_Value' and 'Dosage_Unit'
[ACTION] Split 'Duration' into 'Duration_Days' and 'Duration_Unit'
[ACTION] Split 'Symptoms' into 7 binary indicator columns
[ACTION] Split 'Drug_Allergies' into 3 binary indicator columns
[ACTION] Split 'Chronic_Conditions' into 4 binary indicator columns
[ACTION] Split 'Genetic_Disorders' into 3 binary indicator columns
[ACTION] Preserved 'None' as explicit category in relevant columns
[ACTION] Dropped leakage columns: ['Treatment_Effectiveness', 'Adverse_Reactions']
[ACTION] Coerced 7 numeric columns
[INFO] Validation passed: Target column has no missing values
[INFO] Validation passed: Patient

---

## 5. Tổng kết & Kiểm tra chất lượng

Đánh giá kết quả cleaning của toàn bộ 4 dataset trước khi chuyển sang bước preparing.

In [6]:
# ============================================
# 5. OVERALL QUALITY REPORT
# ============================================
print("="*60)
print("DATA CLEANING - OVERALL REPORT")
print("="*60)

datasets = {
    "CBC": OUTPUT_DIR / "CBC_cleaned.csv",
    "Diseases_and_Symptoms": OUTPUT_DIR / "Diseases_and_Symptoms_cleaned.csv",
    "Laboratory_Data": OUTPUT_DIR / "Laboratory_Data_cleaned.csv",
    "Personalized_Medication": OUTPUT_DIR / "Personalized_Medication_cleaned.csv",
}

report = {}
for name, path in datasets.items():
    if not path.exists():
        print(f"[WARN] {name}: file not found at {path}")
        continue
    df = pd.read_csv(path)
    report[name] = {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_pct": round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
        "duplicate_rows": int(df.duplicated().sum()),
    }
    print(f"\n{name}:")
    print(f"  Rows: {report[name]['rows']:,}")
    print(f"  Columns: {report[name]['columns']}")
    print(f"  Missing cells: {report[name]['missing_cells']:,} ({report[name]['missing_pct']}%)")
    print(f"  Duplicate rows: {report[name]['duplicate_rows']}")

# Save report
report_path = OUTPUT_DIR / "cleaning_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "="*60)
print(f"Report saved to: {report_path}")
print("="*60)

# ============================================
# NEXT STEPS
# ============================================
print("\nNEXT STEPS:")
print("  1. Review cleaning_report.json for data quality metrics.")
print("  2. Proceed to 'preparing.ipynb' for imputation, encoding, scaling.")
print("  3. Consult domain expert for:")
print("     - Outlier flag thresholds (clinical plausibility)")
print("     - 'None' semantics in Personalized Medication")
print("     - BMI mismatch root cause (rounding vs manual entry)")
print("  4. Define train/val/test split strategy (stratified).")
print("  5. Baseline modeling with appropriate algorithms per dataset.")
print("="*60)

DATA CLEANING - OVERALL REPORT

CBC:
  Rows: 9,165
  Columns: 41
  Missing cells: 20,940 (5.57%)
  Duplicate rows: 0

Diseases_and_Symptoms:
  Rows: 246,945
  Columns: 147
  Missing cells: 0 (0.0%)
  Duplicate rows: 113554

Laboratory_Data:
  Rows: 12,009
  Columns: 29
  Missing cells: 0 (0.0%)
  Duplicate rows: 0

Personalized_Medication:
  Rows: 1,000
  Columns: 30
  Missing cells: 1,167 (3.89%)
  Duplicate rows: 0

Report saved to: ..\Dataset\cleaned\cleaning_report.json

NEXT STEPS:
  1. Review cleaning_report.json for data quality metrics.
  2. Proceed to 'preparing.ipynb' for imputation, encoding, scaling.
  3. Consult domain expert for:
     - Outlier flag thresholds (clinical plausibility)
     - 'None' semantics in Personalized Medication
     - BMI mismatch root cause (rounding vs manual entry)
  4. Define train/val/test split strategy (stratified).
  5. Baseline modeling with appropriate algorithms per dataset.
